Code created by: Lorena Espinosa, Johana Rátiva, Eduards Chipatecua 

Class: Procesamiento del Lenguaje Natural

University: Universidad de los Andes

Date: August 23, 2026

In [6]:
import xml.etree.ElementTree as ET
import re
import unicodedata
import math
import sys
import spacy
import pandas as pd
import numpy as np
import nltk
import import_ipynb
import Metricas_Evaluacion as metricas

from IPython.display import display
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from pathlib import Path
from rank_bm25 import BM25Okapi
from gensim import corpora, models, similarities

stemmer = PorterStemmer()
for recurso in ["punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

# Motores de Busqueda

## Procesamiento DataSet

### Lectura de documentos

el contenido empezaba con el titulo del documento, lo cual puede resultar en busquedas dobles en elementos donde solo esta una vez, por eso convienve filtrarlo desde la lectura

In [7]:
def read_naf_document(path):
    """ 
    Esta función lee un documento en formato NAF (Natural Annotation Format) desde la ruta especificada
    y extrae información relevante del mismo a partir de su estructura XML.
    Devuelve un diccionario con el 
        ID del documento, 
        el título y 
        el contenido del texto.

    Se limpia el contenido eliminando el título del mismo si este existe,
    y se eliminan los caracteres de puntuación y espacios en blanco al inicio del contenido.
    """
    tree = ET.parse(path) # abre el archivo NAF y lo parsea en un árbol XML
    root = tree.getroot() # obtiene la raíz del árbol XML 

    header = root.find("nafHeader") # acceso a los hijos del elemento raíz
    file_desc = header.find("fileDesc")
    public = header.find("public")
    raw = root.find("raw")
    titulo = file_desc.get("title")
    contenido = raw.text

    if titulo is not None:
        contenido = contenido[len(titulo):].lstrip(". \n")
        return {
            "id": public.get("publicId"),
            "titulo": titulo,
            "contenido": contenido}
    else:
        return {
            "id": public.get("publicId"),
            "contenido": contenido}

docs_path = Path("docs-raw-texts")
naf_files = list(docs_path.glob("*.naf"))
documents = []

for file_path in naf_files:
    document = read_naf_document(file_path)
    documents.append(document)

In [8]:
documents[0]

{'id': 'd001',
 'titulo': 'William Beaumont and the Human Digestion',
 'contenido': 'William Beaumont: Physiology of digestion Image Source.  On November 21, 1785, US-American\xa0surgeon William Beaumont was born. He became best known as “Father of Gastric Physiology” following his research on human\xa0digestion. William Beaumont was born in Lebanon, Connecticut and became a physician. He served as a surgeon’s mate\xa0in the Army\xa0during the War of 1812. He opened a private practice\xa0in Plattsburgh, New York, but rejoined the Army\xa0as a surgeon in 1819. Beaumont was stationed at Fort Mackinac on Mackinac Island in Michigan in the early 1820s when it existed to protect the interests of the American Fur Company. The fort became the refuge\xa0for a wounded 19-year-old French-Canadian fur trader named\xa0Alexis St. Martin\xa0when a shotgun went off by accident in the American Fur Company store at close range June 6th, 1822.\xa0St. Martin’s wound was quite serious because his\xa0stoma

### Pre-Procesamiento.Limpieza y normalizacion

In [9]:
def clean_text(text):
    """ 
    Esta función limpia el texto de un documento eliminando etiquetas HTML y URLs.
    Además, reemplaza múltiples espacios en blanco por un solo espacio y elimina los espacios al inicio 
    y al final del texto.
    """
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(text, remove_accents=False):
    """ 
    Esta función normaliza el texto de un documento, 
    convirtiéndolo a minúsculas y eliminando acentos si es necesario.
    """
    text = unicodedata.normalize("NFKC", text).lower()
    if remove_accents:
        text = "".join(
            char
            for char in unicodedata.normalize("NFD", text)
            if unicodedata.category(char) != "Mn"
        )
    return text
for document in documents:
    texto = document["titulo"] + " " + document["contenido"]
    texto = clean_text(texto)
    document["contenido_limpio"] = normalize_text(texto, True)

In [10]:
documents[0]

{'id': 'd001',
 'titulo': 'William Beaumont and the Human Digestion',
 'contenido': 'William Beaumont: Physiology of digestion Image Source.  On November 21, 1785, US-American\xa0surgeon William Beaumont was born. He became best known as “Father of Gastric Physiology” following his research on human\xa0digestion. William Beaumont was born in Lebanon, Connecticut and became a physician. He served as a surgeon’s mate\xa0in the Army\xa0during the War of 1812. He opened a private practice\xa0in Plattsburgh, New York, but rejoined the Army\xa0as a surgeon in 1819. Beaumont was stationed at Fort Mackinac on Mackinac Island in Michigan in the early 1820s when it existed to protect the interests of the American Fur Company. The fort became the refuge\xa0for a wounded 19-year-old French-Canadian fur trader named\xa0Alexis St. Martin\xa0when a shotgun went off by accident in the American Fur Company store at close range June 6th, 1822.\xa0St. Martin’s wound was quite serious because his\xa0stoma

### Tokenizacion
Se seleccionó spaCy como tokenizer porque su segmentación permite separar términos unidos por signos de puntuación, como guiones, y conservar sus componentes como tokens independientes. Esto resulta conveniente para la recuperación booleana mediante índice invertido, ya que permite que términos individuales como jean o nicolas puedan ser indexados y recuperados independientemente.en la tabla se puede ver las evidencias de eso

In [11]:
def tokenize_nltk(text):
    return word_tokenize(text)

nlp = spacy.load("en_core_web_sm")
def tokenize_spacy(text):
    doc = nlp(text)
    return [token.text for token in doc]

In [12]:
print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [13]:
nlp("Ayuda a entender")

Ayuda a entender

In [14]:
def compare_tokens(tokens_nltk, tokens_spacy):
    """
    Esta función compara dos listas de tokens generadas por NLTK y spaCy, respectivamente.
    Devuelve dos listas:
    - Tokens que están presentes en la lista de NLTK pero no en la de spaCy.
    - Tokens que están presentes en la lista de spaCy pero no en la de NLTK.
    """
    set_nltk = set(tokens_nltk)
    set_spacy = set(tokens_spacy)

    solo_nltk = [token for token in tokens_nltk if token not in set_spacy]
    solo_spacy = [token for token in tokens_spacy if token not in set_nltk]

    return solo_nltk, solo_spacy

resultados = []

for document in documents:
    texto = document["contenido_limpio"]

    tokens_nltk = tokenize_nltk(texto)
    tokens_spacy = tokenize_spacy(texto)

    set_nltk = set(tokens_nltk)
    set_spacy = set(tokens_spacy)

    solo_nltk = [token for token in tokens_nltk if token not in set_spacy]
    solo_spacy = [token for token in tokens_spacy if token not in set_nltk]

    resultados.append({
        "id": document["id"],
        "titulo": document["titulo"],
        "tokens_nltk": len(tokens_nltk),
        "tokens_spacy": len(tokens_spacy),
        "diferencia": len(tokens_nltk) - len(tokens_spacy),
        "solo_NLTK": solo_nltk,
        "solo_spaCy": solo_spacy
    })

comparacion = pd.DataFrame(resultados)

display(comparacion)

,id,titulo,tokens_nltk,tokens_spacy,diferencia,solo_NLTK,solo_spaCy
0,d001,William Beaumont and the Human Digestion,484,497,-13,"[us-american, ’, s, 1812., 1819., 19-year-old,...","[us, -, ’s, 1812, 1819, 19, -, year, -, old, f..."
1,d002,Selma Lagerlöf and the wonderful Adventures of...,516,513,3,"[s, 1890., s, 1900., s, s]","[’s, 1890, book‘s, 1900, ’s, ’s]"
2,d003,Ferdinand de Lesseps and the Suez Canal,994,998,-4,"[1805-1894, ‘, s, career-diplomats, vice-consu...","[-, 1894, lessep‘s, -, diplomats, vice, -, con..."
3,d004,Walt Disney’s ‘Steamboat Willie’ and the Rise ...,559,560,-1,"[s, s, black-and-white, s, s, s, s, sound-on-f...","[’s, ’s, black, -, -, white, ’s, ’s, ’s, ’s, -..."
4,d005,Eugene Wigner and the Structure of the Atomic ...,694,696,-2,"[1902-1995, ’, s, ’, s, ’, s, x-ray, s, d-matr...","[1995, ’s, ’s, ’s, x, ray, hilbert‘s, d, matri..."
...,...,...,...,...,...,...,...
326,d327,James Parkinson and Parkinson’s Disease,557,553,4,"[’, s, s, 1886., s, s, under-privileged, post-...","[’s, parkinson‘s, 1886, parkinson‘s, parkinson..."
327,d328,Juan de la Cierva and the Autogiro,896,896,0,"[oct., 1925., single-rotor, today, ‘, s, [, 1,...","[oct, single, -, today‘s, it.[1, fixed, -, rot..."
328,d329,Squire Whipple – The Father of the Iron Bridge,482,484,-2,"[us-american, cotton-spinning, ’, s, short-lif...","[us, -, american, cotton, -, spinning, ’s, sho..."
329,d330,William Playfair and the Beginnings of Infogra...,927,920,7,"[’, s, trade-balance, time-series, mill-wright...","[’s, -, balance, -, mill, -, wright, playfair‘..."


### Limpieza de tokens
Quitamos palabras de parada y puntuacion en los tockes 

In [15]:
for document in documents:
    document["tokens"] = [ token for token in nlp(document["contenido_limpio"]) if not token.is_punct and not token.is_stop ]

In [16]:
documents[0]

{'id': 'd001',
 'titulo': 'William Beaumont and the Human Digestion',
 'contenido': 'William Beaumont: Physiology of digestion Image Source.  On November 21, 1785, US-American\xa0surgeon William Beaumont was born. He became best known as “Father of Gastric Physiology” following his research on human\xa0digestion. William Beaumont was born in Lebanon, Connecticut and became a physician. He served as a surgeon’s mate\xa0in the Army\xa0during the War of 1812. He opened a private practice\xa0in Plattsburgh, New York, but rejoined the Army\xa0as a surgeon in 1819. Beaumont was stationed at Fort Mackinac on Mackinac Island in Michigan in the early 1820s when it existed to protect the interests of the American Fur Company. The fort became the refuge\xa0for a wounded 19-year-old French-Canadian fur trader named\xa0Alexis St. Martin\xa0when a shotgun went off by accident in the American Fur Company store at close range June 6th, 1822.\xa0St. Martin’s wound was quite serious because his\xa0stoma

### Steamming
Elegimos utilizar Porter Stemmer porque queremos que palabras que representan una misma idea, pero que aparecen en diferentes formas, puedan ser tratadas como un mismo término durante la búsqueda. Por ejemplo, engine y engines se convierten en engin, mientras que digestion y digestive se convierten en digest. Aunque el resultado no siempre sea una palabra real, esto no es un problema para nuestro motor, porque el stem se utiliza únicamente como término interno del índice. Así podemos aumentar las posibilidades de encontrar documentos relevantes aunque la consulta y el documento utilicen formas diferentes de una palabra.

Adicionalmente el proceso de stemming es menos costoso que el proceso de lematización dado que no requiere un recurso linguistico para generar el corte en la palabra.

In [17]:
for document in documents:
    document["tokens"] = [stemmer.stem(token.text) for token in document["tokens"]]

In [18]:
documents[0]

{'id': 'd001',
 'titulo': 'William Beaumont and the Human Digestion',
 'contenido': 'William Beaumont: Physiology of digestion Image Source.  On November 21, 1785, US-American\xa0surgeon William Beaumont was born. He became best known as “Father of Gastric Physiology” following his research on human\xa0digestion. William Beaumont was born in Lebanon, Connecticut and became a physician. He served as a surgeon’s mate\xa0in the Army\xa0during the War of 1812. He opened a private practice\xa0in Plattsburgh, New York, but rejoined the Army\xa0as a surgeon in 1819. Beaumont was stationed at Fort Mackinac on Mackinac Island in Michigan in the early 1820s when it existed to protect the interests of the American Fur Company. The fort became the refuge\xa0for a wounded 19-year-old French-Canadian fur trader named\xa0Alexis St. Martin\xa0when a shotgun went off by accident in the American Fur Company store at close range June 6th, 1822.\xa0St. Martin’s wound was quite serious because his\xa0stoma

In [19]:
print("Número de documentos:", len(documents))
documentos_sin_tokens = [
    document["id"]
    for document in documents
    if not document["tokens"]
]

print("Documentos sin tokens:", documentos_sin_tokens)
cantidad_tokens = [
    len(document["tokens"])
    for document in documents
]

print("Mínimo:", min(cantidad_tokens))
print("Máximo:", max(cantidad_tokens))
print("Promedio:", sum(cantidad_tokens) / len(cantidad_tokens))

Número de documentos: 331
Documentos sin tokens: []
Mínimo: 125
Máximo: 701
Promedio: 342.7794561933535


## Recuperación booleana usando índice invertido (BSII).

### Creacion indice inverso

In [20]:
def build_inverted_index(documents):
    """
    Esta función construye un índice invertido a partir de una lista de documentos.
    Cada documento es un diccionario que contiene un identificador único y una lista de tokens.
    Devuelve un diccionario donde las claves son los tokens y los valores son listas de identificadores de documentos que contienen ese token.
    
    Parameters:
    documents (list): Una lista de diccionarios, donde cada diccionario representa un documento y contiene un identificador único y una lista de tokens.
    Returns:
    dict: Un diccionario que representa el índice invertido, donde las claves son los tokens y
    los valores son listas de identificadores de documentos que contienen ese token.
    """
    index = {}
    for document in documents:
        document_id = document["id"]
        for token in document["tokens"]:
            if token not in index:
                index[token] = set()
            index[token].add(document_id)
    for token in index:
        index[token] = sorted(index[token])
    return index

inverted_index = build_inverted_index(documents)
tabla_indice = pd.DataFrame(
    [
        {
            "termino": termino,
            "documentos": ", ".join(sorted(documentos))
        }
        for termino, documentos in list(inverted_index.items())[:20]
    ]
)

display(tabla_indice)

,termino,documentos
0,william,"d001, d009, d015, d028, d035, d055, d056, d069..."
1,beaumont,d001
2,human,"d001, d002, d007, d012, d032, d036, d046, d050..."
3,digest,"d001, d049, d054, d102, d263, d314"
4,physiolog,"d001, d037, d046, d062, d113, d120, d133, d191..."
5,imag,"d001, d004, d008, d015, d017, d018, d020, d021..."
6,sourc,"d001, d015, d025, d033, d041, d045, d053, d080..."
7,novemb,"d001, d002, d003, d004, d005, d006, d007, d008..."
8,21,"d001, d032, d069, d072, d099, d102, d128, d138..."
9,1785,"d001, d026, d043, d074, d086, d184, d245, d288..."


In [21]:
inverted_index.keys()

dict_keys(['william', 'beaumont', 'human', 'digest', 'physiolog', 'imag', 'sourc', 'novemb', '21', '1785', 'american', 'surgeon', 'born', 'best', 'known', 'father', 'gastric', 'follow', 'research', 'lebanon', 'connecticut', 'physician', 'serv', 'mate', 'armi', 'war', '1812', 'open', 'privat', 'practic', 'plattsburgh', 'new', 'york', 'rejoin', '1819', 'station', 'fort', 'mackinac', 'island', 'michigan', 'earli', '1820', 'exist', 'protect', 'interest', 'fur', 'compani', 'refug', 'wound', '19', 'year', 'old', 'french', 'canadian', 'trader', 'name', 'alexi', 'st', 'martin', 'shotgun', 'went', 'accid', 'store', 'close', 'rang', 'june', '6th', '1822', 'stomach', 'perfor', 'rib', 'broken', 'expect', 'young', 'man', 'surviv', 'skin', 'fuse', 'hole', 'leav', 'perman', 'fistula', '1', 'quickli', 'notic', 'potenti', 'system', 'order', 'gain', 'inform', 'perform', 'numer', 'experi', 'period', 'uncomfort', 'insert', 'bit', 'differ', 'food', 'tie', 'string', 'pull', 'observ', 'remov', 'juic', 'exami

In [22]:
memoria_indice = sys.getsizeof(inverted_index)

for termino, postings in inverted_index.items():
    memoria_indice += sys.getsizeof(termino)
    memoria_indice += sys.getsizeof(postings)

    for documento in postings:
        memoria_indice += sys.getsizeof(documento)

print("Número de documentos:", len(documents))
print("Tamaño del vocabulario:", len(inverted_index))
print("Memoria aproximada del índice:", memoria_indice, "bytes")

Número de documentos: 331
Tamaño del vocabulario: 13923
Memoria aproximada del índice: 6216254 bytes


In [23]:
termino_mas_frecuente = None
termino_menos_frecuente = None

max_postings = 0
min_postings = None

for termino, postings in inverted_index.items():
    cantidad_postings = len(postings)

    if cantidad_postings > max_postings:
        max_postings = cantidad_postings
        termino_mas_frecuente = termino

    if min_postings is None or cantidad_postings < min_postings:
        min_postings = cantidad_postings
        termino_menos_frecuente = termino

print("Término más frecuente:", termino_mas_frecuente)
print("Número de postings:", max_postings)

print("Término menos frecuente:", termino_menos_frecuente)
print("Número de postings:", min_postings)

Término más frecuente: yovisto
Número de postings: 320
Término menos frecuente: beaumont
Número de postings: 1


### Consultas booleanas


Sin *skip pointers*, la intersección tiene complejidad $O(m+n)$ ya que el algortimo que se usa es un *merge* de las listas de postings asociadas a los terminos. Con *skip pointers* se pueden reducir las comparaciones  procesando las intersecciones en menos tiempo al saltar segmentos que no pueden producir coincidencias; sin embargo, el beneficio depende de la distribución y longitud de las listas, por lo que no debe asumirse una mejora asintótica fija para todos los casos. En este caso, tomando $\sqrt{p}$ *skip pointers* distribuidos uniformemente, en el mejor de los casos la complejidad se reduce a:


$$
O(\sqrt{m} + \sqrt{n})
$$


Los skip pointers no son útiles para las consultas OR, ya que en este caso no buscamos únicamente documentos que aparezcan en ambas listas, sino que necesitamos conservar todos los documentos presentes en cualquiera de ellas, eliminando solamente los repetidos. Por esta razón, saltar posiciones mediante skip pointers podría hacer que se omitan documentos que deberían formar parte del resultado.


In [24]:
def build_skip_pointers(postings):
    """
    Construye punteros de salto para una lista de postings.

    Los punteros de salto aprovechan la estructura ordenada de los
    postings para permitir saltos durante la intersección de listas,
    mejorando la eficiencia de las búsquedas.

    La longitud del salto se determina utilizando la raíz cuadrada
    del tamaño de la lista de postings.

    Parameters:
        postings (list): Lista ordenada de identificadores de documentos.

    Returns:
        dict: Diccionario donde las claves son los índices de los
            postings y los valores son los índices de destino de los saltos.
    """
    skips = {}
    skip_length = int(math.sqrt(len(postings)))
    if skip_length < 2:
        return skips

    for i in range(0, len(postings) - skip_length, skip_length):
        skips[i] = i + skip_length

    return skips


def intersect_without_skips(postings1, postings2):
    """ 
    Compara dos lista: postings1 y postings2 a partir de punteros mientras existan elementos que comparar en ambas listas. 
    Iteración i y j sean menores a la cantidad de obtenos dentro del posting 1 y 2 respectivamente. 

    La función va contando las comparaciones y consolidando las intersecciones en result.

    Parameters:
        postings1: Lista de documentos ordenados en los que aparece un termino dado 
        postings2: Lista de documentos ordenados en los que aparece el termino comparable 

    Returns:
        result: Lista de documentos ordenados en los que aparecen ambos terminos
        comparisons: Número de comparaciones realizadas para evaluar el consumo sin punteros de salto.   

    """
    result = []
    comparisons = 0
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        comparisons += 1
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            i += 1
        else:
            j += 1
    return result, comparisons

def union_postings(postings1, postings2):
    """
    Esta función une los documentos de los postings garantizando la no duplicidad de los mismos.
    Sirve para responder querys de OR. En donde se espera el retorno de documentos con al menos uno de los terminos consultados.


    Parameters:
        postings1: Lista de documentos ordenados en los que aparece un termino dado 
        postings2: Lista de documentos ordenados en los que aparece el termino comparable 
        
    Returns:
        result: Lista de todos documentos ordenados sin duplicados.
    """
    result = []
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            result.append(postings1[i])
            i += 1
        else:
            result.append(postings2[j])
            j += 1
    while i < len(postings1):
        result.append(postings1[i])
        i += 1

    while j < len(postings2):
        result.append(postings2[j])
        j += 1

    return result

def not_postings(postings, documents):
    """
    Esta función busca entregar los documentos que no contienen al termino.
    
    Parameters:
        postings: Lista de documentos ordenados en los que aparece un termino dado.
        documents: Lista de documentos del corpus
    
    Returns:
        result: Lista de documenos ordenados que no contienen el termino.

    """
    result = []
    documentos_postings = set(postings)
    for document in documents:
        if document["id"] not in documentos_postings:
            result.append(document["id"])
    result.sort()
    return result

def intersect_with_skips(postings1, postings2, skips1, skips2):
    """Calcula la intersección de dos listas de postings ordenadas utilizando skip pointers para mejorar la eficiencia de la búsqueda.

    Durante la intersección, compara los documentos de ambas listas mediante dos punteros.
    Cuando el documento de una lista es menor que el de la otra, intenta utilizar un skip pointer para avanzar varias posiciones.
    El salto solo se realiza si su documento de destino no supera el documento actual de la otra lista, evitando así omitir posibles coincidencias.

    Parameters:
        postings1: Primera lista de identificadores de documentos, ordenada de forma ascendente.
        postings2: Segunda lista de identificadores de documentos, ordenada de forma ascendente.
        skips1: Diccionario de skip pointers para postings1. Las claves son índices de la lista y los valores son los índices de destino del salto.
        skips2: Diccionario de skip pointers para postings2. Las claves son índices de la lista y los valores son los índices de destino del salto.

    Returns:
        tuple: Una tupla con tres elementos:
            - result (list): Lista ordenada de documentos presentes en ambas posting lists.
            - comparisons (int): Número de comparaciones realizadas durante la intersección.
            - skips_used (int): Número de skip pointers utilizados."""
    result = []
    comparisons = 0
    skips_used = 0
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        comparisons += 1
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            if (i in skips1 and postings1[skips1[i]] <= postings2[j]):
                i = skips1[i]
                skips_used += 1
            else:
                i += 1
        else:
            if (j in skips2 and postings2[skips2[j]] <= postings1[i]):
                j = skips2[j]
                skips_used += 1
            else:
                j += 1
    return result, comparisons, skips_used

### Evaluacion Queries

#### Consulta booleana

In [25]:
def split_boolean_query(query):
    """
    Divide una consulta booleana en tokens individuales, separando explícitamente los paréntesis para facilitar su posterior procesamiento.

    Parameters:
        query (str): Consulta booleana ingresada como una cadena de texto. Puede contener términos, operadores booleanos (AND, OR, NOT) y paréntesis.

    Returns:
        list: Lista de tokens obtenidos a partir de la consulta."""
    query = query.replace("(", " ( ")
    query = query.replace(")", " ) ")

    return query.split()

def process_boolean_tokens(tokens):
    """Preprocesa los tokens de una consulta booleana antes de evaluarla.

    Los operadores booleanos y los paréntesis se conservan sin modificar.
    Los términos de búsqueda se limpian, normalizan, tokenizan, filtran
    para eliminar signos de puntuación y palabras vacías, y finalmente
    se reducen mediante stemming para que sean compatibles con los
    términos utilizados en el índice invertido.

    Parameters:
        tokens (list): Lista de tokens obtenidos a partir de una consulta
            booleana.

    Returns:
        list: Lista de tokens procesados, conservando los operadores
            booleanos y paréntesis."""
    
    result = []
    operators = ["AND", "OR", "NOT", "(", ")"]
    for token in tokens:
        if token in operators:
            result.append(token)
        else:
            content = clean_text(token)
            contentN = normalize_text(content, True)
            processed_tokens = [ t.text for t in nlp(contentN) if not t.is_punct and not t.is_stop]
            for processed_token in processed_tokens:result.append(stemmer.stem(processed_token))
    return result

def parse_not(tokens, position):
    """Evalúa expresiones que contienen operadores NOT, términos individuales
    o expresiones agrupadas mediante paréntesis.

    El operador NOT se evalúa de forma recursiva y obtiene el complemento
    de la posting list correspondiente respecto al conjunto de documentos.
    Cuando encuentra un paréntesis, delega la evaluación de la expresión
    interna al parser de OR.

    Parameters:
        tokens (list): Lista de tokens de la consulta booleana procesada.
        position (int): Posición actual dentro de la lista de tokens.

    Returns:
        tuple: Una tupla formada por:
            - result (list): Posting list resultante de la expresión evaluada.
            - position (int): Posición del siguiente token que debe ser
              procesado."""
    
    if tokens[position] == "NOT":
        result, position = parse_not(tokens, position + 1)
        return not_postings(result, documents), position
    if tokens[position] == "(":
        result, position = parse_or(tokens, position + 1)
        return result, position + 1
    return inverted_index.get(tokens[position], []), position + 1

def parse_and(tokens, position):
    """Evalúa expresiones booleanas que contienen el operador AND.

    Obtiene inicialmente una expresión mediante parse_not y posteriormente
    realiza intersecciones sucesivas cuando encuentra operadores AND.
    Para cada intersección construye skip pointers para ambas posting lists
    y utiliza dichos punteros para mejorar la eficiencia de la intersección.

    Parameters:
        tokens (list): Lista de tokens de la consulta booleana procesada.
        position (int): Posición actual dentro de la lista de tokens.

    Returns:
        tuple: Una tupla formada por:
            - result (list): Posting list resultante de la intersección.
            - position (int): Posición del siguiente token que debe ser
              procesado."""
    
    result, position = parse_not(tokens, position)
    while position < len(tokens) and tokens[position] == "AND":
        next_result, position = parse_not(tokens, position + 1)
        skips1 = build_skip_pointers(result)
        skips2 = build_skip_pointers(next_result)
        result, _, _ = intersect_with_skips(result, next_result, skips1, skips2)
    return result, position

def parse_or(tokens, position):
    """Evalúa expresiones booleanas que contienen el operador OR.

    Obtiene inicialmente una expresión mediante parse_and y posteriormente
    realiza uniones sucesivas cuando encuentra operadores OR. La función
    delega la evaluación de las expresiones AND a parse_and, estableciendo
    así la precedencia de AND sobre OR.

    Parameters:
        tokens (list): Lista de tokens de la consulta booleana procesada.
        position (int): Posición actual dentro de la lista de tokens.

    Returns:
        tuple: Una tupla formada por:
            - result (list): Posting list resultante de la unión.
            - position (int): Posición del siguiente token que debe ser
              procesado."""
    result, position = parse_and(tokens, position)
    while position < len(tokens) and tokens[position] == "OR":
        next_result, position = parse_and(tokens, position + 1)
        result = union_postings(result, next_result)
    return result, position

def evaluate_boolean_query(tokens):
    """Evalúa una consulta booleana completa a partir de sus tokens procesados.

    Inicia el análisis sintáctico desde la primera posición de la consulta
    utilizando parse_or, que a su vez coordina la evaluación de las
    expresiones AND, OR, NOT y los paréntesis.

    Parameters:
        tokens (list): Lista de tokens de una consulta booleana procesada.

    Returns:
        list: Lista ordenada de identificadores de los documentos que
            satisfacen la consulta booleana."""
    result, _ = parse_or(tokens, 0)
    return result

In [26]:

process_boolean_tokens(split_boolean_query("river AND (french OR method)"))

['river', 'AND', '(', 'french', 'OR', 'method', ')']

In [27]:
pruebas = [
    "river AND french",
    "river OR french",
    "NOT river",
    "river AND NOT french",
    "river OR NOT french",
    "river OR french AND method",
    "river AND (french OR method)",
    "(river OR french) AND method"
]

for query in pruebas:
    tokens = split_boolean_query(query)
    tokens = process_boolean_tokens(tokens)

    resultado = evaluate_boolean_query(tokens)

    print(query)
    print("Tokens:", tokens)
    print("Cantidad:", len(resultado))
    print("Resultado:", resultado)
    print()

river AND french
Tokens: ['river', 'AND', 'french']
Cantidad: 7
Resultado: ['d029', 'd053', 'd096', 'd184', 'd189', 'd257', 'd313']

river OR french
Tokens: ['river', 'OR', 'french']
Cantidad: 94
Resultado: ['d001', 'd003', 'd004', 'd011', 'd012', 'd014', 'd019', 'd025', 'd026', 'd029', 'd035', 'd038', 'd042', 'd047', 'd052', 'd053', 'd056', 'd063', 'd066', 'd074', 'd075', 'd078', 'd083', 'd085', 'd090', 'd095', 'd096', 'd099', 'd101', 'd105', 'd108', 'd109', 'd111', 'd120', 'd122', 'd126', 'd135', 'd138', 'd139', 'd140', 'd141', 'd145', 'd147', 'd148', 'd150', 'd152', 'd157', 'd160', 'd162', 'd164', 'd166', 'd167', 'd168', 'd170', 'd181', 'd184', 'd185', 'd188', 'd189', 'd190', 'd191', 'd202', 'd212', 'd231', 'd232', 'd235', 'd236', 'd237', 'd242', 'd246', 'd248', 'd252', 'd257', 'd263', 'd264', 'd265', 'd268', 'd280', 'd281', 'd283', 'd285', 'd288', 'd293', 'd302', 'd304', 'd309', 'd313', 'd317', 'd323', 'd324', 'd325', 'd327', 'd330', 'd331']

NOT river
Tokens: ['NOT', 'river']
Cant

river OR french AND method
Tokens: ['river', 'OR', 'french', 'AND', 'method']
Cantidad: 45
Resultado: ['d004', 'd012', 'd025', 'd026', 'd029', 'd035', 'd042', 'd052', 'd053', 'd078', 'd085', 'd090', 'd095', 'd096', 'd099', 'd109', 'd122', 'd138', 'd139', 'd141', 'd152', 'd157', 'd170', 'd184', 'd189', 'd191', 'd202', 'd212', 'd231', 'd235', 'd236', 'd246', 'd257', 'd263', 'd265', 'd280', 'd281', 'd285', 'd288', 'd302', 'd309', 'd313', 'd327', 'd330', 'd331']

river AND (french OR method)
Tokens: ['river', 'AND', '(', 'french', 'OR', 'method', ')']
Cantidad: 9
Resultado: ['d029', 'd035', 'd053', 'd096', 'd184', 'd189', 'd231', 'd257', 'd313']

(river OR french) AND method
Tokens: ['(', 'river', 'OR', 'french', ')', 'AND', 'method']
Cantidad: 21
Resultado: ['d012', 'd026', 'd035', 'd042', 'd052', 'd085', 'd099', 'd109', 'd122', 'd170', 'd202', 'd212', 'd231', 'd235', 'd246', 'd263', 'd281', 'd285', 'd288', 'd327', 'd330']



####  ¿Cómo se compara el algoritmo de mezcla con y sin skip pointers para la consulta early AND telecommunication?

Para la consulta **`early AND telecommunication`**, ambos algoritmos recuperan los mismos documentos: `['d060', 'd100', 'd231']`. Sin embargo, el algoritmo **sin skip pointers** realiza **107 comparaciones**, mientras que el algoritmo **con skip pointers** realiza únicamente **41 comparaciones**, utilizando **6 saltos**. Esto representa una reducción de aproximadamente **61.7 % en el número de comparaciones**, mostrando que los *skip pointers* permiten evitar recorrer posiciones innecesarias de las listas de postings y hacen más eficiente la intersección en este caso.


In [69]:
term1 = stemmer.stem("early")
term2 = stemmer.stem("telecommunication")
p1 = inverted_index.get(term1, [])
p2 = inverted_index.get(term2, [])

print("early:", p1)
print("telecommunication:", p2)

print(term1)
print(term2)

print("early:", p1)
print("telecommunication:", p2)

early: ['d001', 'd003', 'd009', 'd014', 'd015', 'd016', 'd017', 'd018', 'd021', 'd022', 'd023', 'd024', 'd025', 'd027', 'd029', 'd034', 'd035', 'd039', 'd045', 'd046', 'd048', 'd052', 'd054', 'd055', 'd056', 'd057', 'd058', 'd060', 'd061', 'd063', 'd065', 'd066', 'd068', 'd069', 'd071', 'd073', 'd074', 'd076', 'd077', 'd080', 'd081', 'd085', 'd086', 'd091', 'd093', 'd095', 'd097', 'd100', 'd101', 'd107', 'd109', 'd110', 'd113', 'd115', 'd118', 'd122', 'd124', 'd126', 'd129', 'd130', 'd131', 'd132', 'd133', 'd135', 'd136', 'd137', 'd138', 'd141', 'd142', 'd144', 'd146', 'd148', 'd151', 'd152', 'd154', 'd159', 'd167', 'd168', 'd171', 'd172', 'd173', 'd174', 'd175', 'd185', 'd190', 'd192', 'd193', 'd194', 'd198', 'd199', 'd201', 'd203', 'd204', 'd205', 'd209', 'd211', 'd212', 'd214', 'd215', 'd216', 'd218', 'd219', 'd221', 'd223', 'd229', 'd230', 'd231', 'd232', 'd233', 'd234', 'd235', 'd237', 'd240', 'd241', 'd244', 'd247', 'd248', 'd249', 'd250', 'd251', 'd255', 'd257', 'd259', 'd262', 

In [75]:
resultado_sin, comparaciones_sin = intersect_without_skips(
    p1,
    p2
)

skips_p1 = build_skip_pointers(p1)
skips_p2 = build_skip_pointers(p2)

resultado_con, comparaciones_con, skips_utilizados = intersect_with_skips(
    p1,
    p2,
    skips_p1,
    skips_p2
)

print("Resultado sin skips:", resultado_sin)
print("Resultado con skips:", resultado_con)

print("Comparaciones sin skips:", comparaciones_sin)
print("Comparaciones con skips:", comparaciones_con)

print("Skip pointers utilizados:", skips_utilizados)

print("Reducción al usar skips:", ((107 - 41) / 107) * 100, "%")

Resultado sin skips: ['d060', 'd100', 'd231']
Resultado con skips: ['d060', 'd100', 'd231']
Comparaciones sin skips: 107
Comparaciones con skips: 41
Skip pointers utilizados: 6
Reducción al usar skips: 61.6822429906542 %


#### Consultas binarias AND

In [28]:
queries_path = Path("queries-raw-texts")
query_files = list(queries_path.glob("*.naf"))

queries = []

for file_path in query_files:
    query = read_naf_document(file_path)
    queries.append(query)

In [29]:
queries[:5]

[{'id': 'q01', 'contenido': 'Fabrication of music instruments'},
 {'id': 'q02', 'contenido': 'famous German poetry'},
 {'id': 'q03', 'contenido': 'Romanticism'},
 {'id': 'q04', 'contenido': 'University of Edinburgh research'},
 {'id': 'q06', 'contenido': 'bridge construction'}]

In [30]:
for query in queries:

    content = clean_text(query["contenido"])
    contentN = normalize_text( content,True)
    tokens = [
        token.text
        for token in nlp(contentN)
        if not token.is_punct and not token.is_stop
    ]
    query["stems"] = [ stemmer.stem(token) for token in tokens]

In [31]:
resultados_queries = []

for query in queries:

    terms = query["stems"]

    if not terms:
        resultados_queries.append({
            "query": query["id"],
            "terminos": "",
            "resultado": [],
            "comparaciones_sin_skips": 0,
            "comparaciones_con_skips": 0,
            "skips_utilizados": 0,
            "iguales": True
        })
        continue

    resultado_sin = inverted_index.get(terms[0], [])
    comparaciones_sin = 0

    for term in terms[1:]:
        postings = inverted_index.get(term, [])
        resultado_sin, comparaciones = intersect_without_skips(
            resultado_sin,
            postings
        )
        comparaciones_sin += comparaciones

        if not resultado_sin:
            break

    resultado_con = inverted_index.get(terms[0], [])
    comparaciones_con = 0
    skips_utilizados = 0

    for term in terms[1:]:

        postings = inverted_index.get(term, [])

        skips_resultado = build_skip_pointers(resultado_con)
        skips_postings = build_skip_pointers(postings)

        resultado_con, comparaciones, skips = intersect_with_skips(
            resultado_con,
            postings,
            skips_resultado,
            skips_postings
        )

        comparaciones_con += comparaciones
        skips_utilizados += skips

        if not resultado_con:
            break

    resultados_queries.append({
        "query": query["id"],
        "terminos": " AND ".join(terms),
        "resultado": resultado_con,
        "comparaciones_sin_skips": comparaciones_sin,
        "comparaciones_con_skips": comparaciones_con,
        "skips_utilizados": skips_utilizados,
        "iguales": resultado_sin == resultado_con
    })

tabla_queries = pd.DataFrame(resultados_queries)

display(tabla_queries)

,query,terminos,resultado,comparaciones_sin_skips,comparaciones_con_skips,skips_utilizados,iguales
0,q01,fabric AND music AND instrument,[],34,26,2,True
1,q02,famou AND german AND poetri,"[d291, d293]",257,245,2,True
2,q03,romantic,"[d105, d147, d152, d283, d291, d318]",0,0,0,True
3,q04,univers AND edinburgh AND research,[d286],324,170,14,True
4,q06,bridg AND construct,"[d026, d029, d069, d257, d297, d303, d329]",68,56,2,True
5,q07,walk AND fame AND star,"[d004, d034]",32,32,0,True
6,q08,scientist AND work AND atom AND bomb,"[d108, d110, d117, d205, d251]",411,403,1,True
7,q09,invent AND internet,"[d198, d205, d223]",53,25,4,True
8,q10,earli AND telecommun AND method,[d231],158,64,10,True
9,q12,explor AND south AND pole,"[d176, d250, d277]",69,65,2,True


In [32]:
tabla_queries_ordenada = tabla_queries.copy()
numeros = []

for query_id in tabla_queries_ordenada["query"]:
    numeros.append(int(query_id[1:]))

tabla_queries_ordenada["numero_query"] = numeros
tabla_queries_ordenada = tabla_queries_ordenada.sort_values("numero_query")
output_path = Path("BSII-AND-queries_results")

with output_path.open("w", encoding="utf-8") as output_file:
    for _, row in tabla_queries_ordenada.iterrows():
        documents_found = sorted(row["resultado"])
        output_file.write(
            f'{row["query"]} {",".join(documents_found)}\n'
        )

print(f"Archivo generado: {output_path.resolve()}")

Archivo generado: C:\Users\adwar\Documents\repositorios\Learning\NLP-ml\Tarea1\BSII-AND-queries_results


## Recuperación ranqueada y vectorización de documentos (RRDV)

### Representación vectorial ponderada tf.idf

La estrategia utiliza el índice invertido para obtener el DF de cada término y limitar el cálculo del TF a los documentos en los que el término aparece. Esto evita calcular el TF para documentos en los que el término no está presente y aprovecha la estructura construida para BSII. Sin embargo, el índice invertido almacena únicamente los identificadores de los documentos, por lo que la frecuencia del término debe calcularse nuevamente a partir de los tokens de cada documento. Además, para localizar cada documento asociado a un posting, la implementación recorre la colección de documentos. Por esta razón, aunque se reduce el número de documentos sobre los que se calcula el TF, la implementación no es completamente eficiente

In [33]:
def build_tfidf_matrix(inverted_index, documents):
    """Construye la matriz TF-IDF utilizando el índice invertido para obtener
    el DF y calcular el peso de cada término en los documentos donde aparece.
    
    Parameters:
        inverted_index: Diccionario del Índice invertido en donde relaciona cada término con la lista de identificadores de los documentos en los que aparece.
        documents: Lista de documentos que contiene sus identificadores y tokens, utilizados para calcular la frecuencia del término (TF).
    Returns:
        tf_idf_matriz: Matriz TF-IDF representada como un diccionario donde cada término contiene los documentos en los que aparece y su respectivo peso TF-IDF.
    """
    tf_idf_matrix = {}

    for term in inverted_index:
        tf_idf_matrix[term] = {}
        df = len(inverted_index[term])
        idf = np.log10(len(documents) / df)

        for document_id in inverted_index[term]:
            for document in documents:
                if document["id"] == document_id:
                    tf = document["tokens"].count(term)
                    tf_idf_matrix[term][document_id] = np.log10(1 + tf) * idf
                    break

    return tf_idf_matrix     

 
tf_idf_matrix = build_tfidf_matrix(inverted_index, documents)

In [34]:
def build_document_vectors(tf_idf_matrix, documents):
    """
    Construye un vector TF-IDF para cada documento a partir de la matriz TF-IDF, asignando cero a los términos que no aparecen en el documento.

    Parameters:
        tf_idf_matrix: Diccionario de la Matriz TF-IDF representada como un diccionario
            de términos, documentos y sus respectivos pesos.
        documents: Lista de documentos con sus identificadores.

    Returns:
        dict: Diccionario que relaciona cada identificador de documento con su vector TF-IDF.
    """
    document_vectors = {}

    for document in documents:
        document_id = document["id"]
        vector = []
        for term in tf_idf_matrix:
            vector.append(tf_idf_matrix[term].get(document_id, 0))
        document_vectors[document_id] = vector

    return document_vectors

document_vectors = build_document_vectors(tf_idf_matrix,documents)

def build_query_vector(query, tf_idf_matrix, inverted_index, documents):
    """Construye el vector TF-IDF de una consulta utilizando el mismo
    vocabulario y esquema de ponderación empleado para los documentos.

    Parameters:
        query: Diccionario de la consulta procesada que contiene los términos normalizados o stemmizados.
        tf_idf_matrix: Diccionario Matriz TF-IDF utilizada para definir el vocabulario y los pesos de los términos.
        inverted_index: Diccionario Índice invertido utilizado para obtener el DF de cada término de la consulta.
        documents : Lista de documentos utilizada para calcular el IDF.

    Returns:
        list: Vector TF-IDF correspondiente a la consulta."""
    
    query_vector = []
    for term in tf_idf_matrix:
        tf = query["stems"].count(term)
        if tf > 0:
            df = len(inverted_index[term])
            idf = np.log10(len(documents) / df)
            tf_idf = np.log10(1 + tf) * idf
        else:
            tf_idf = 0
        query_vector.append(tf_idf)
    return query_vector
    

# vector ara uno de los queries
query_vector = build_query_vector(
    query,
    tf_idf_matrix,
    inverted_index,
    documents
)
print("Dimensión:", len(query_vector))
print("Primeros 10 valores:", query_vector[:10])

    

Dimensión: 13923
Primeros 10 valores: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [35]:

ids = list(document_vectors.keys())[:10]

# Terminos que tienen algun peso en esos documentos
terms = []

for term in tf_idf_matrix:
    if any(tf_idf_matrix[term].get(doc_id, 0) > 0 for doc_id in ids):
        terms.append(term)

# Construimos la tabla
vector_table = pd.DataFrame(
    {
        term: [tf_idf_matrix[term].get(doc_id, 0) for doc_id in ids]
        for term in terms
    },
    index=ids
)

vector_table = vector_table.T
display(vector_table)

,d001,d002,d003,d004,d005,d006,d007,d008,d009,d010
william,0.638752,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.247103,0.000000
beaumont,2.806946,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
human,0.353870,0.223267,0.0,0.0,0.0,0.0,0.223267,0.0,0.000000,0.000000
digest,1.879585,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
physiolog,1.006969,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
protein,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.614916
dynam,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.504144
soul,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.486687
implic,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.548133


### Similitud del coseno

In [36]:
def cosine_similarity(vector1, vector2):
    """Calcula la similitud coseno entre dos vectores y retorna un valor que representa su similitud.
    
    Parameters:
        vector1: Lista primer vector numérico.
        vector2: Lista segundo vector numérico.

    Returns:
        float: Similitud coseno entre los dos vectores. Retorna 0 si
            alguno de los vectores tiene norma cero."""
    dot_product = np.dot(vector1, vector2)
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot_product / (norm1 * norm2)

d1 = document_vectors["d096"]
d2 = document_vectors["d169"]

similarity = cosine_similarity(d1, d2)

print("Similitud:", similarity)

Similitud: 0.01940197047531942


In [37]:
def rank_documents(query,document_vectors,tf_idf_matrix,inverted_index,documents):
    """
    Ordena los documentos según su similitud coseno con una consulta, colocando primero los documentos con mayor similitud.

    Parameters:
        query (dict): Consulta procesada que contiene sus términos.
        document_vectors (dict): Vectores TF-IDF de los documentos.
        tf_idf_matrix (dict): Matriz TF-IDF utilizada para construir el vector de la consulta.
        inverted_index (dict): Índice invertido utilizado para obtener el DF de los términos.
        documents (list): Lista de documentos utilizada para calcular el IDF.

    Returns:
        list: Lista de tuplas con el identificador del documento y su
            similitud con la consulta, ordenada de mayor a menor similitud.
    """
    query_vector = build_query_vector(query,tf_idf_matrix,inverted_index,documents)
    rankings = []

    for document_id, document_vector in document_vectors.items():

        similarity = cosine_similarity(query_vector,document_vector)

        if similarity > 0:
            rankings.append((document_id, similarity))

    return sorted(rankings,key=lambda x: x[1],reverse=True)

In [38]:
coseno_rankings = {}

for query in queries:
    coseno_rankings[query["id"]] = rank_documents(query,document_vectors,tf_idf_matrix,inverted_index,documents)

In [39]:
filas = []

for query_id, ranking in coseno_rankings.items():

    for posicion, (document_id, similarity) in enumerate(ranking[:10], start=1):

        filas.append({
            "query": query_id,
            "posición": posicion,
            "documento": document_id,
            "similitud": similarity
        })

tabla_ranking = pd.DataFrame(filas)

display(tabla_ranking)

,query,posición,documento,similitud
0,q01,1,d259,0.109798
1,q01,2,d085,0.087924
2,q01,3,d186,0.075858
3,q01,4,d254,0.072904
4,q01,5,d016,0.071303
...,...,...,...,...
341,q46,6,d091,0.081280
342,q46,7,d081,0.081024
343,q46,8,d121,0.077181
344,q46,9,d013,0.076337


In [40]:
query_id = "q06"

tabla_q06 = pd.DataFrame(
    coseno_rankings[query_id][:10],
    columns=["documento", "similitud"]
)

tabla_q06.insert(0, "posición", range(1, len(tabla_q06) + 1))

display(tabla_q06)

,posición,documento,similitud
0,1,d329,0.230026
1,2,d297,0.224940
2,3,d026,0.160921
3,4,d029,0.120839
4,5,d233,0.106594
5,6,d025,0.104681
6,7,d257,0.103213
7,8,d069,0.065687
8,9,d186,0.045624
9,10,d077,0.043958


In [41]:
ids = ["d329", "d297", "d026","d029","d233"]

for document in documents:
    if document["id"] in ids:
        print("=" * 80)
        print("ID:", document["id"])
        print("Título:", document["titulo"])
        print("Contenido:", document["contenido_limpio"])

ID: d026
Título: Jean-Rondolphe Perronet and the Bridges of Paris
Contenido: jean-rondolphe perronet and the bridges of paris jean-rodolphe perronet (1708-1794). on october 27, 1708, french architect and structural engineer jean-rodolphe perronet was born. he is best known for his many stone arch bridges, among them his most popular work, the paris pont de la concorde. jean-rodolphe perronet was born in suresnes, a suburb of paris, the son of a swiss guardsman. at 17 he entered the architectural practice of jean beausire, “first architect” to the city of paris, as an apprentice. he was put in charge of the design and construction of paris’s grand sewer, embankment works and the maintenance of the banlieue’s roads. in 1735, he was named sous-ingenieur (under-engineer) to alencon. perronet’s perceived energy for the district of alencon came to the notice of trudaine – the overseer of finances in charge of roads – who put him in charge of training surveyors and those drawing maps to provi

### BM25

In [42]:
corpus = [document["tokens"] for document in documents]

bm25 = BM25Okapi(
    corpus,
    k1=1.5,
    b=0.75
)
bm25_rankings = {}
filas = []
for query in queries:
    scores = bm25.get_scores(query["stems"])
    bm25_rankings[query["id"]] = [(documents[i]["id"], scores[i]) for i in np.argsort(scores)[::-1] if scores[i] > 0]

for query_id, ranking in bm25_rankings.items():

    for posicion, (document_id, score) in enumerate(ranking[:10], start=1):

        filas.append({
            "query": query_id,
            "posición": posicion,
            "documento": document_id,
            "BM25": score
        })

tabla_bm25 = pd.DataFrame(filas)

display(tabla_bm25)

,query,posición,documento,BM25
0,q01,1,d085,8.132767
1,q01,2,d254,7.971316
2,q01,3,d186,7.405890
3,q01,4,d016,7.022649
4,q01,5,d259,6.726422
...,...,...,...,...
341,q46,6,d145,8.482019
342,q46,7,d117,8.258585
343,q46,8,d247,8.187469
344,q46,9,d094,8.172796


Al variar b entre 0 y 1 se observa que la normalización por longitud afecta el ranking de BM25. Para q06, con b=0, d297 ocupa el primer lugar y d329 el segundo; al aumentar b hasta 0.50, sus posiciones se intercambian y d329 permanece primero hasta b=1. Otros documentos, como d026, mantienen su posición durante toda la variación. Esto muestra que el efecto de b depende de las características de cada documento y que una mayor normalización por longitud puede modificar el orden de documentos con puntajes similares.

In [43]:
"Prueba de variacion de b "
query = next(q for q in queries if q["id"] == "q06")

resultados_bm25 = []

for b in [0, 0.25, 0.5, 0.75, 1]:

    bm25 = BM25Okapi(
        corpus,
        k1=1.5,
        b=b
    )

    scores = bm25.get_scores(query["stems"])
    ranking = [(documents[i]["id"], scores[i])for i in np.argsort(scores)[::-1] if scores[i] > 0]

    for posicion, (documento, score) in enumerate(ranking[:10], 1):

        resultados_bm25.append({
            "b": b,
            "posición": posicion,
            "documento": documento,
            "score": score
        })


In [44]:
tabla_bm25 = pd.DataFrame(resultados_bm25)

tabla_bm25["resultado"] = (
    tabla_bm25["documento"] 
    + " (" 
    + tabla_bm25["score"].round(2).astype(str) 
    + ")"
)

tabla_bm25 = tabla_bm25.pivot(
    index="posición",
    columns="b",
    values="resultado"
)

display(tabla_bm25)

b,0.00,0.25,0.50,0.75,1.00
posición,,,,,
1,d297 (9.54),d297 (9.62),d329 (9.72),d329 (9.83),d329 (9.96)
2,d329 (9.49),d329 (9.6),d297 (9.71),d297 (9.8),d297 (9.89)
3,d026 (9.43),d026 (9.38),d026 (9.34),d026 (9.29),d026 (9.24)
4,d257 (7.63),d257 (7.5),d257 (7.38),d257 (7.26),d029 (7.3)
5,d029 (6.99),d029 (7.06),d029 (7.14),d029 (7.22),d257 (7.15)
6,d025 (6.03),d025 (6.04),d025 (6.04),d025 (6.05),d025 (6.05)
7,d233 (5.48),d233 (5.54),d233 (5.61),d233 (5.67),d233 (5.73)
8,d303 (4.52),d069 (4.65),d069 (4.8),d069 (4.95),d069 (5.12)
9,d069 (4.52),d303 (4.15),d303 (3.84),d303 (3.57),d004 (3.49)


In [45]:
ids = ["d329", "d297", "d026","d029","d233"]

for document in documents:
    if document["id"] in ids:
        print("=" * 80)
        print("ID:", document["id"])
        print("Título:", document["titulo"])
        print("Contenido:", document["contenido_limpio"])

ID: d026
Título: Jean-Rondolphe Perronet and the Bridges of Paris
Contenido: jean-rondolphe perronet and the bridges of paris jean-rodolphe perronet (1708-1794). on october 27, 1708, french architect and structural engineer jean-rodolphe perronet was born. he is best known for his many stone arch bridges, among them his most popular work, the paris pont de la concorde. jean-rodolphe perronet was born in suresnes, a suburb of paris, the son of a swiss guardsman. at 17 he entered the architectural practice of jean beausire, “first architect” to the city of paris, as an apprentice. he was put in charge of the design and construction of paris’s grand sewer, embankment works and the maintenance of the banlieue’s roads. in 1735, he was named sous-ingenieur (under-engineer) to alencon. perronet’s perceived energy for the district of alencon came to the notice of trudaine – the overseer of finances in charge of roads – who put him in charge of training surveyors and those drawing maps to provi

### Creacion de archivos de resultados

In [46]:
with open("RRDV-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query in queries:

        resultados = rank_documents(query,document_vectors,tf_idf_matrix, inverted_index,documents)

        resultados_formateados = [ f"{document_id}: {similitud:.6f}"  for document_id, similitud in resultados ]

        archivo.write(
            f"{query['id']} {','.join(resultados_formateados)}\n"
        )

In [47]:
with open("BM25-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query_id, ranking in bm25_rankings.items():

        resultados = [f"{document_id}: {score:.6f}" for document_id, score in ranking]

        archivo.write(
            f"{query_id} {','.join(resultados)}\n"
        )

### Evaluacion de metricas (Resultados)

In [48]:
relevance_judgments = {}

with open("relevance-judgments.tsv", "r", encoding="utf-8") as archivo:
    for linea in archivo:
        query_id, documentos = linea.strip().split("\t")
        relevance_judgments[query_id] = {}
        for documento in documentos.split(","):
            document_id, relevancia = documento.split(":")
            relevance_judgments[query_id][document_id] = int(relevancia)

In [49]:
def evaluate_ranking(ranking, relevance_judgments):
    """
    Evalúa el ranking de documentos de cada consulta mediante P@M,
    R@M y NDCG@M, utilizando los juicios de relevancia correspondientes.

    Parameters:
        ranking (dict): Ranking de documentos por consulta con sus scores.
        relevance_judgments (dict): Juicios de relevancia por consulta.

    Returns:
        pd.DataFrame: Resultados de evaluación por consulta.
    """

    resultados = []

    for query_id, ranking_query in ranking.items():

        juicios = relevance_judgments.get(query_id, {})
        M = len(juicios)

        documentos = [document_id for document_id, score in ranking_query]

        relevancia_binaria = [ 1 if document_id in juicios else 0 for document_id in documentos]

        relevancia_graduada = [juicios.get(document_id, 0) for document_id in documentos]

        resultados.append({
            "query": query_id,
            "M": M,
            "P@M": metricas.precision_at_k(relevancia_binaria, M),
            "R@M": metricas.recall_at_k(relevancia_binaria, M, M),
            "NDCG@M": metricas.ndcg_at_k(
                relevancia_graduada, M, "linear"
            )
        })

    return pd.DataFrame(resultados)

#### BM25

In [50]:
resultados_bm25 = evaluate_ranking(
    bm25_rankings,
    relevance_judgments
)

display(resultados_bm25)

,query,M,P@M,R@M,NDCG@M
0,q01,3,0.666667,0.666667,0.507615
1,q02,11,0.545455,0.545455,0.584844
2,q03,6,1.000000,1.000000,0.989525
3,q04,7,0.857143,0.857143,0.890951
4,q06,6,0.666667,0.666667,0.826491
5,q07,4,0.250000,0.250000,0.509097
6,q08,12,0.750000,0.750000,0.886503
7,q09,6,1.000000,1.000000,0.917885
8,q10,8,0.375000,0.375000,0.477187
9,q12,4,0.750000,0.750000,0.775845


In [51]:
tabla_evaluacion = resultados_bm25.copy()

tabla_evaluacion[["P@M", "R@M", "NDCG@M"]] = (
    tabla_evaluacion[["P@M", "R@M", "NDCG@M"]]
    .round(4)
)

display(tabla_evaluacion)

,query,M,P@M,R@M,NDCG@M
0,q01,3,0.6667,0.6667,0.5076
1,q02,11,0.5455,0.5455,0.5848
2,q03,6,1.0000,1.0000,0.9895
3,q04,7,0.8571,0.8571,0.8910
4,q06,6,0.6667,0.6667,0.8265
5,q07,4,0.2500,0.2500,0.5091
6,q08,12,0.7500,0.7500,0.8865
7,q09,6,1.0000,1.0000,0.9179
8,q10,8,0.3750,0.3750,0.4772
9,q12,4,0.7500,0.7500,0.7758


#### Coseno

In [52]:
resultados_coseno = evaluate_ranking(
    coseno_rankings,
    relevance_judgments
)

display(resultados_coseno)

,query,M,P@M,R@M,NDCG@M
0,q01,3,0.333333,0.333333,0.196954
1,q02,11,0.545455,0.545455,0.568734
2,q03,6,1.000000,1.000000,0.995943
3,q04,7,0.714286,0.714286,0.781593
4,q06,6,0.666667,0.666667,0.851844
5,q07,4,0.250000,0.250000,0.321204
6,q08,12,0.750000,0.750000,0.860365
7,q09,6,0.833333,0.833333,0.818389
8,q10,8,0.375000,0.375000,0.472121
9,q12,4,0.750000,0.750000,0.775845


In [53]:
resultados_coseno.columns

Index(['query', 'M', 'P@M', 'R@M', 'NDCG@M'], dtype='str')

In [54]:
tabla_evaluacion_coseno = resultados_coseno.copy()

tabla_evaluacion_coseno[["P@M", "R@M", "NDCG@M"]] = (
    tabla_evaluacion_coseno[["P@M", "R@M", "NDCG@M"]]
    .round(4)
)

display(tabla_evaluacion_coseno)

,query,M,P@M,R@M,NDCG@M
0,q01,3,0.3333,0.3333,0.1970
1,q02,11,0.5455,0.5455,0.5687
2,q03,6,1.0000,1.0000,0.9959
3,q04,7,0.7143,0.7143,0.7816
4,q06,6,0.6667,0.6667,0.8518
5,q07,4,0.2500,0.2500,0.3212
6,q08,12,0.7500,0.7500,0.8604
7,q09,6,0.8333,0.8333,0.8184
8,q10,8,0.3750,0.3750,0.4721
9,q12,4,0.7500,0.7500,0.7758


In [55]:
comparacion = pd.DataFrame({
    "query": resultados_coseno["query"],
    "M": resultados_coseno["M"],
    "P@M Coseno": resultados_coseno["P@M"],
    "P@M BM25": resultados_bm25["P@M"],
    "R@M Coseno": resultados_coseno["R@M"],
    "R@M BM25": resultados_bm25["R@M"],
    "NDCG@M Coseno": resultados_coseno["NDCG@M"],
    "NDCG@M BM25": resultados_bm25["NDCG@M"]
})

display(comparacion.round(4))

,query,M,P@M Coseno,P@M BM25,R@M Coseno,R@M BM25,NDCG@M Coseno,NDCG@M BM25
0,q01,3,0.3333,0.6667,0.3333,0.6667,0.1970,0.5076
1,q02,11,0.5455,0.5455,0.5455,0.5455,0.5687,0.5848
2,q03,6,1.0000,1.0000,1.0000,1.0000,0.9959,0.9895
3,q04,7,0.7143,0.8571,0.7143,0.8571,0.7816,0.8910
4,q06,6,0.6667,0.6667,0.6667,0.6667,0.8518,0.8265
5,q07,4,0.2500,0.2500,0.2500,0.2500,0.3212,0.5091
6,q08,12,0.7500,0.7500,0.7500,0.7500,0.8604,0.8865
7,q09,6,0.8333,1.0000,0.8333,1.0000,0.8184,0.9179
8,q10,8,0.3750,0.3750,0.3750,0.3750,0.4721,0.4772
9,q12,4,0.7500,0.7500,0.7500,0.7500,0.7758,0.7758


In [56]:
evaluacion_rrdv = evaluate_ranking(coseno_rankings,relevance_judgments)
evaluacion_bm25 = evaluate_ranking(bm25_rankings,relevance_judgments)

In [57]:
map_cos = metricas.mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in coseno_rankings.items()
    ]
)

map_bm25 = metricas.mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in bm25_rankings.items()
    ]
)

print("MAP COS:", map_cos)
print("MAP BM25:", map_bm25)

MAP COS: 0.7047680299424705
MAP BM25: 0.7543733244416239


## Recuperación ranqueada con GENSIM

In [58]:
corpus = [document["tokens"] for document in documents]
dictionary = corpora.Dictionary(corpus)

corpus_bow = [dictionary.doc2bow(document["tokens"])for document in documents]

print("Tamaño del vocabulario - GENSIM:", len(dictionary))

tfidf = models.TfidfModel(corpus_bow,id2word=dictionary,smartirs="ltc")
corpus_tfidf = tfidf[corpus_bow]
index = similarities.SparseMatrixSimilarity(corpus_tfidf,num_features=len(dictionary),num_best=None)

gensim_rankings = {}

for query in queries:
    query_bow = dictionary.doc2bow(query["stems"])
    query_tfidf = tfidf[query_bow]
    scores = index[query_tfidf]
    ranking = [(documents[i]["id"], scores[i]) for i in np.argsort(scores)[::-1] if scores[i] > 0]
    gensim_rankings[query["id"]] = ranking

Tamaño del vocabulario - GENSIM: 13923


In [59]:
with open("GENSIM-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query_id, ranking in gensim_rankings.items():

        resultados = [
            f"{document_id}: {score:.6f}"
            for document_id, score in ranking
        ]

        archivo.write(
            f"{query_id} {','.join(resultados)}\n"
        )

In [60]:
evaluacion_gensim = evaluate_ranking(gensim_rankings,relevance_judgments)

In [61]:
map_gensim = metricas.mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in gensim_rankings.items()
    ]
)
print("MAP GENSIM:", map_gensim)

MAP GENSIM: 0.7068766149575881


## Comparativa de metricas Finales

#### Tabla Comparativa metricas

In [62]:
evaluacion_comparativa = evaluacion_rrdv.merge(
    evaluacion_bm25,
    on=["query", "M"],
    suffixes=("_COS", "_BM25")
)

evaluacion_comparativa = evaluacion_comparativa.merge(
    evaluacion_gensim,
    on=["query", "M"]
)

evaluacion_comparativa = evaluacion_comparativa.rename(
    columns={
        "P@M": "P@M_GENSIM",
        "R@M": "R@M_GENSIM",
        "NDCG@M": "NDCG@M_GENSIM"
    }
)

evaluacion_comparativa["numero_query"] = (
    evaluacion_comparativa["query"].str[1:].astype(int)
)

evaluacion_comparativa = evaluacion_comparativa.sort_values(
    "numero_query"
).drop(columns="numero_query")

# evaluacion_comparativa = evaluacion_comparativa[
#     [
#         "query",
#         "M",
#         "P@M_COS",
#         "P@M_BM25",
#         "P@M_GENSIM",
#         "R@M_COS",
#         "R@M_BM25",
#         "R@M_GENSIM",
#         "NDCG@M_COS",
#         "NDCG@M_BM25",
#         "NDCG@M_GENSIM"
#     ]
# ]


evaluacion_comparativa = evaluacion_comparativa[
    [
        "query",
        "M",
        "P@M_GENSIM",
        "R@M_GENSIM",
        "NDCG@M_BM25",
        "NDCG@M_GENSIM"
    ]
]

display(evaluacion_comparativa.round(4))

,query,M,P@M_GENSIM,R@M_GENSIM,NDCG@M_BM25,NDCG@M_GENSIM
0,q01,3,0.3333,0.3333,0.5076,0.1970
1,q02,11,0.5455,0.5455,0.5848,0.5650
2,q03,6,1.0000,1.0000,0.9895,0.9959
3,q04,7,0.7143,0.7143,0.8910,0.7816
4,q06,6,0.8333,0.8333,0.8265,0.8497
5,q07,4,0.2500,0.2500,0.5091,0.3212
6,q08,12,0.7500,0.7500,0.8865,0.8582
7,q09,6,0.8333,0.8333,0.9179,0.8184
8,q10,8,0.3750,0.3750,0.4772,0.4683
9,q12,4,0.7500,0.7500,0.7758,0.7758


#### Tabla Comparativa MPA

In [63]:
tabla_map = pd.DataFrame({
    "Función de ranqueo": [
        "COS",
        "BM25",
        "GENSIM"
    ],
    "MAP": [
        map_cos,
        map_bm25,
        map_gensim
    ]
})

display(tabla_map)

,Función de ranqueo,MAP
0,COS,0.704768
1,BM25,0.754373
2,GENSIM,0.706877
